In [0]:
--Días con Sismos Significativos en Perú
SELECT 
  `FECHA`,
  `DEPARTAMENTO`,
  `MAGNITUD_MAXIMA`,
  `PROFUNDIDAD_PROMEDIO_KM`,
  `SISMOS_SIGNIFICATIVOS`
FROM `big_data_team1`.`saori`.`igp_gold_fia`
WHERE `SISMOS_SIGNIFICATIVOS` > 0
ORDER BY `FECHA` DESC;

-- Total de Sismos por Departamento y Año
SELECT 
  `DEPARTAMENTO`,
  YEAR(`FECHA`) AS `AÑO`,
  SUM(`SISMOS_IGP`) AS `TOTAL_SISMOS`
FROM `big_data_team1`.`saori`.`igp_gold_fia`
GROUP BY `DEPARTAMENTO`, YEAR(`FECHA`)
ORDER BY `DEPARTAMENTO`, `AÑO`;

--Top 10 Departamentos con Más Damnificados por Sismos
SELECT 
  `DEPARTAMENTO`,
  SUM(`DAMNIFICADOS`) AS `TOTAL_DAMNIFICADOS`,
  SUM(`PERSONAS_AFECTADAS`) AS `TOTAL_PERSONAS_AFECTADAS`,
  SUM(`VIVIENDAS_DESTRUIDAS`) AS `TOTAL_VIVIENDAS_DESTRUIDAS`,
  SUM(`COSTO_AYUDA`) AS `COSTO_AYUDA_TOTAL`
FROM `big_data_team1`.`saori`.`igp_gold_fia`
GROUP BY `DEPARTAMENTO`
ORDER BY `TOTAL_DAMNIFICADOS` DESC
LIMIT 10

--Comparación de Actividad Sísmica por Año (2018-2020)
SELECT 
  YEAR(`FECHA`) AS `AÑO`,
  SUM(`SISMOS_IGP`) AS `TOTAL_SISMOS`,
  MAX(`MAGNITUD_MAXIMA`) AS `MAGNITUD_MAXIMA`,
  AVG(`PROFUNDIDAD_PROMEDIO_KM`) AS `PROFUNDIDAD_PROMEDIO_KM`
FROM `big_data_team1`.`saori`.`igp_gold_fia`
WHERE YEAR(`FECHA`) IN (2018, 2019, 2020)
GROUP BY YEAR(`FECHA`)
ORDER BY `AÑO`

--Cantidad de Sismos por Mes Durante el Período 2018-2020
SELECT 
  YEAR(`FECHA`) AS `AÑO`,
  MONTH(`FECHA`) AS `MES`,
  SUM(`SISMOS_IGP`) AS `TOTAL_SISMOS`
FROM `big_data_team1`.`saori`.`igp_gold_fia`
GROUP BY YEAR(`FECHA`), MONTH(`FECHA`)
ORDER BY YEAR(`FECHA`), MONTH(`FECHA`)

--Análisis de Tendencia y Proyección para 2021
SELECT 
  `AÑO`,
  SUM(`TOTAL_SISMOS`) AS `TOTAL_ANUAL`,
  ROUND(AVG(`TOTAL_SISMOS`), 2) AS `PROMEDIO_MENSUAL`,
  ROUND(STDDEV(`TOTAL_SISMOS`), 2) AS `DESVIACION_ESTANDAR`,
  MIN(`TOTAL_SISMOS`) AS `MIN_MENSUAL`,
  MAX(`TOTAL_SISMOS`) AS `MAX_MENSUAL`
FROM (
  SELECT 
    YEAR(`FECHA`) AS `AÑO`,
    MONTH(`FECHA`) AS `MES`,
    SUM(`SISMOS_IGP`) AS `TOTAL_SISMOS`
  FROM `big_data_team1`.`saori`.`igp_gold_fia`
  WHERE YEAR(`FECHA`) IN (2018, 2019, 2020)
  GROUP BY YEAR(`FECHA`), MONTH(`FECHA`)
) subquery
GROUP BY `AÑO`
ORDER BY `AÑO`

--Top 5 Departamentos con Mayor Riesgo de Actividad Sísmica Fuerte
SELECT 
  `DEPARTAMENTO`,
  `TOTAL_SISMOS`,
  `TOTAL_SISMOS_SIGNIFICATIVOS`,
  `MAGNITUD_MAXIMA_REGISTRADA`,
  `DIAS_CON_SISMOS_FUERTES`,
  `PROFUNDIDAD_PROMEDIO`,
  `TOTAL_DAMNIFICADOS`,
  ROUND(
    (`TOTAL_SISMOS_SIGNIFICATIVOS` * 3.0) + 
    (`DIAS_CON_SISMOS_FUERTES` * 2.0) + 
    ((`MAGNITUD_MAXIMA_REGISTRADA` - 5.0) * 10.0) +
    (CASE WHEN `PROFUNDIDAD_PROMEDIO` < 70 THEN 10 ELSE 0 END) +
    (`TOTAL_SISMOS` / 10.0)
  , 2) AS `INDICE_RIESGO`
FROM (
  SELECT 
    `DEPARTAMENTO`,
    SUM(`SISMOS_IGP`) AS `TOTAL_SISMOS`,
    SUM(`SISMOS_SIGNIFICATIVOS`) AS `TOTAL_SISMOS_SIGNIFICATIVOS`,
    MAX(`MAGNITUD_MAXIMA`) AS `MAGNITUD_MAXIMA_REGISTRADA`,
    COUNT(DISTINCT CASE WHEN `MAGNITUD_MAXIMA` >= 5.0 THEN `FECHA` END) AS `DIAS_CON_SISMOS_FUERTES`,
    ROUND(AVG(`PROFUNDIDAD_PROMEDIO_KM`), 2) AS `PROFUNDIDAD_PROMEDIO`,
    SUM(`DAMNIFICADOS`) AS `TOTAL_DAMNIFICADOS`
  FROM `big_data_team1`.`saori`.`igp_gold_fia`
  GROUP BY `DEPARTAMENTO`
) subquery
ORDER BY `INDICE_RIESGO` DESC
LIMIT 5

--Proyección de Damnificados y Costo de Ayuda para 2021
SELECT 
  YEAR(`FECHA`) AS `AÑO`,
  SUM(`DAMNIFICADOS`) AS `TOTAL_DAMNIFICADOS`,
  SUM(`PERSONAS_AFECTADAS`) AS `TOTAL_PERSONAS_AFECTADAS`,
  SUM(`VIVIENDAS_DESTRUIDAS`) AS `TOTAL_VIVIENDAS_DESTRUIDAS`,
  SUM(`VIVIENDAS_AFECTADAS`) AS `TOTAL_VIVIENDAS_AFECTADAS`,
  ROUND(SUM(`COSTO_AYUDA`), 2) AS `COSTO_AYUDA_TOTAL`,
  SUM(`EMERGENCIAS_SISMO`) AS `TOTAL_EMERGENCIAS`,
  SUM(`FALLECIDOS`) AS `TOTAL_FALLECIDOS`,
  SUM(`HERIDOS`) AS `TOTAL_HERIDOS`
FROM `big_data_team1`.`saori`.`igp_gold_fia`
WHERE YEAR(`FECHA`) IN (2018, 2019, 2020)
GROUP BY YEAR(`FECHA`)
ORDER BY `AÑO`